In [ ]:
#!pip install keras-tuner

In [1]:
import keras
from keras import layers
import keras_tuner as kt
from sklearn.datasets import fetch_california_housing, load_iris
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, accuracy_score

from tensorflow import keras
from keras import layers
import keras_tuner as kt

from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd


https://keras.io/keras_tuner/getting_started/

In [2]:
hp = kt.HyperParameters()
# Load dataset
def load_dataset():
    data = fetch_california_housing()
    X, y = data.data, data.target

    return X, y

In [3]:
# Preprocessing
scaler = StandardScaler()

def preprocess_data(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    return X_train, X_test, y_train, y_test

In [4]:
# Build Keras model with tunable hyperparameters
def build_model(hp, input_shape):

    # Define Hyperparams

    # Tune the activation function to use.
    activation=hp.Choice("activation", ["relu", "tanh"])
    # The number of hidden layers
    num_layers = hp.Int('num_layers', 2, 4)
    # Define the optimizer learning rate as a hyperparameter.
    learning_rate = hp.Float("lr", min_value=1e-4, max_value=1e-2, sampling="log")


    # Build the model
    model = keras.Sequential()
    # Input Layer
    model.add(layers.Input(shape=(input_shape,)))

    # Hidden Layers
    for i in range(num_layers):
        # Tune number of units.
        units=hp.Int(f'units_{i}', 32, 128, step=32)
        #print(units)
        model.add(layers.Dense(units=units,activation=activation))
        # Dropout - Tune whether to use dropout.
        if hp.Boolean("dropout"):
            model.add(layers.Dropout(rate=0.25))


    # Output Layer
    model.add(layers.Dense(1))

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
                  loss='mse', metrics=['mae'])

    return model

In [5]:
# Run the pipeline
def run_pipeline():
    print('Loading data.......')
    X, y = load_dataset()
    X_train, X_test, y_train, y_test = preprocess_data(X, y)

    print('Tuning parameters.......')

    tuner = kt.RandomSearch(
        lambda hp: build_model(hp, X_train.shape[1]),
        objective='val_loss',
        max_trials=5,
        executions_per_trial=2,
        #directory='./',
        #project_name='regression_model'
    )

    tuner.search(X_train, y_train, epochs=20, validation_split=0.2, verbose=2)
    best_model = tuner.get_best_models(num_models=1)[0]
    best_hp = tuner.get_best_hyperparameters(1)[0]

    print('Training the model ...............')
    best_model.fit(X_train, y_train, epochs=30, validation_split=0.2, verbose=2)
    y_pred = best_model.predict(X_test).flatten()
    #build_model(hp=hp,input_shape=X_train.shape[1])
    return best_model

In [6]:
# Run the pipeline
print("=== REGRESSION: California Housing ===")
model = run_pipeline()

Trial 5 Complete [00h 00m 11s]
val_loss: 0.5017461329698563

Best val_loss So Far: 0.2903352528810501
Total elapsed time: 00h 00m 54s
Training the model ...............
Epoch 1/30


/Users/geethikapanikker/ai-env/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


413/413 - 1s - 1ms/step - loss: 0.2653 - mae: 0.3516 - val_loss: 0.2968 - val_mae: 0.3815
Epoch 2/30
413/413 - 0s - 552us/step - loss: 0.2670 - mae: 0.3522 - val_loss: 0.3015 - val_mae: 0.3645
Epoch 3/30
413/413 - 0s - 553us/step - loss: 0.2703 - mae: 0.3545 - val_loss: 0.2872 - val_mae: 0.3635
Epoch 4/30
413/413 - 0s - 553us/step - loss: 0.2597 - mae: 0.3490 - val_loss: 0.3087 - val_mae: 0.3692
Epoch 5/30
413/413 - 0s - 549us/step - loss: 0.2559 - mae: 0.3439 - val_loss: 0.2958 - val_mae: 0.3753
Epoch 6/30
413/413 - 0s - 556us/step - loss: 0.2521 - mae: 0.3422 - val_loss: 0.2997 - val_mae: 0.3732
Epoch 7/30
413/413 - 0s - 553us/step - loss: 0.2550 - mae: 0.3464 - val_loss: 0.3104 - val_mae: 0.3854
Epoch 8/30
413/413 - 0s - 555us/step - loss: 0.2543 - mae: 0.3431 - val_loss: 0.3020 - val_mae: 0.3752
Epoch 9/30
413/413 - 0s - 550us/step - loss: 0.2485 - mae: 0.3392 - val_loss: 0.2800 - val_mae: 0.3545
Epoch 10/30
413/413 - 0s - 551us/step - loss: 0.2458 - mae: 0.3367 - val_loss: 0.2942 

In [7]:
# Predict for unseen data
new_data = np.array([[8.3252, 41.0, 6.9841, 1.0238, 322.0, 2.5556, 37.88, -122.23]])
# California Housing
new_data_scaled = scaler.transform(new_data)
prediction = model.predict(new_data_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step


array([[4.0376244]], dtype=float32)

In [ ]:
#### Classification

In [8]:
def load_dataset():
    data = pd.read_csv('../../../datasets/pima_indians_diabetes.csv')
    X= data.iloc[:,:8]
    y= data.iloc[:,8]
    return X, y

In [9]:
# Preprocessing
def preprocess_data(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    return X_train, X_test, y_train, y_test

In [10]:
# Build Keras model with tunable hyperparameters
def build_model(hp, input_shape):
    # Hyperparameters
    num_layers = hp.Int('num_layers', 1, 3)
    activation = hp.Choice('activation', ['relu', 'tanh'])
    learning_rate = hp.Choice('lr', [1e-2, 1e-3])

    # Build the network
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_shape,)))


    for i in range(num_layers):
        units = hp.Int(f'units_{i}', 32, 128, step=32)
        model.add(layers.Dense(units=units,
                               activation=activation))
    model.add(layers.Dense(1, activation='sigmoid'))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [11]:
# Run the pipeline
def run_pipeline():
    print('Loading Data.......')
    X, y = load_dataset()
    X_train, X_test, y_train, y_test = preprocess_data(X, y)
    print('Tuning parameters.......')
    tuner = kt.RandomSearch(
        lambda hp: build_model(hp, X_train.shape[1]),
        objective='val_loss',
        max_trials=5,
        executions_per_trial=2,
        #directory='./',
        #project_name='classification_model'
    )

    tuner.search(X_train, y_train, epochs=20, validation_split=0.2, verbose=2)
    best_model = tuner.get_best_models(num_models=1)[0]
    best_hp = tuner.get_best_hyperparameters(1)[0]

    print(f"\n🔧 Best Hyperparameters for Classification:")
    print(best_hp.values)

    print('Training the model')
    # best_model = build_model(best_hp, X_train.shape[1], task, num_classes)
    best_model.fit(X_train, y_train, epochs=30, validation_split=0.2, verbose=2)
    y_pred = best_model.predict(X_test)

    y_pred_class = (y_pred > 0.5).astype(int)
    print("✅ Final Accuracy:", accuracy_score(y_test, y_pred_class))
    return best_model

In [12]:
print("\n=== CLASSIFICATION: Diabetes (Binary) ===")
model = run_pipeline()


=== CLASSIFICATION: Iris (Binary) ===
Loading Data.......
Tuning parameters.......
Reloading Tuner from ./untitled_project/tuner0.json

🔧 Best Hyperparameters for Classification:
{'activation': 'relu', 'num_layers': 3, 'lr': 0.004116480206370245, 'units_0': 96, 'dropout': False, 'units_1': 64, 'units_2': 32}
Training the model
Epoch 1/30


/Users/geethikapanikker/ai-env/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


16/16 - 0s - 31ms/step - accuracy: 0.3992 - loss: 2.7497 - val_accuracy: 0.6829 - val_loss: 0.5950
Epoch 2/30
16/16 - 0s - 3ms/step - accuracy: 0.7332 - loss: 0.5680 - val_accuracy: 0.7236 - val_loss: 0.5767
Epoch 3/30
16/16 - 0s - 3ms/step - accuracy: 0.7658 - loss: 0.5294 - val_accuracy: 0.7398 - val_loss: 0.5142
Epoch 4/30
16/16 - 0s - 3ms/step - accuracy: 0.7658 - loss: 0.4867 - val_accuracy: 0.7317 - val_loss: 0.4900
Epoch 5/30
16/16 - 0s - 3ms/step - accuracy: 0.7760 - loss: 0.4716 - val_accuracy: 0.7561 - val_loss: 0.4784
Epoch 6/30
16/16 - 0s - 3ms/step - accuracy: 0.8004 - loss: 0.4464 - val_accuracy: 0.7724 - val_loss: 0.4745
Epoch 7/30
16/16 - 0s - 3ms/step - accuracy: 0.8126 - loss: 0.4265 - val_accuracy: 0.7805 - val_loss: 0.4740
Epoch 8/30
16/16 - 0s - 3ms/step - accuracy: 0.8208 - loss: 0.4117 - val_accuracy: 0.7642 - val_loss: 0.4687
Epoch 9/30
16/16 - 0s - 3ms/step - accuracy: 0.8269 - loss: 0.3937 - val_accuracy: 0.7480 - val_loss: 0.4741
Epoch 10/30
16/16 - 0s - 3ms/